In [2]:
#basic import 
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns


#models
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


In [3]:
df=pd.read_csv('data/stud.csv')

In [4]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


X and Y valus

In [6]:
X=df.drop(columns=['math_score'])

In [7]:
X.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [8]:
print("category in gender variable",end=" ")
print(df['gender'].unique())

print('category in race_ethincaity variable',end= '')
print(df['race_ethnicity'].unique())

print('category in lunch variable',end= '')
print(df['lunch'].unique())

print('category in test preapartion  variable',end= '')
print(df['test_preparation_course'].unique())

category in gender variable <StringArray>
['female', 'male']
Length: 2, dtype: str
category in race_ethincaity variable<StringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
category in lunch variable<StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
category in test preapartion  variable<StringArray>
['none', 'completed']
Length: 2, dtype: str


In [9]:
y=df['math_score']

In [10]:
y.head()

0    72
1    69
2    90
3    47
4    76
Name: math_score, dtype: int64

In [11]:
#create column tranformation with 3 type of transformation


num_features=X.select_dtypes(exclude="object").columns
cat_features=X.select_dtypes(include="object").columns


print("name features",num_features)
print("category feature",cat_features)

name features Index(['reading_score', 'writing_score'], dtype='str')
category feature Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='str')


C:\Users\hp\AppData\Local\Temp\ipykernel_15284\88041976.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features=X.select_dtypes(include="object").columns


In [12]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",oh_transformer,cat_features),
        ("standardScaling",numeric_transformer,num_features)
    
    ]
)

In [13]:
X=preprocessor.fit_transform(X)

In [14]:
X.shape

(1000, 19)

In [15]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape,X_test.shape

((800, 19), (200, 19))

create an evuate function to give all metrics after model training

In [16]:
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae,mse,rmse,r2_square

In [20]:
models = {
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbor Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "XGBoost Regressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for i in range(len(models)):
    model = list(models.values())[i]   # ✅ fixed .value() → .values()
    
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])   # ✅ fixed indexing
    model_list.append(list(models.keys())[i])

    print('Model performance for training set')

    print(" - RMSE: {:.4f}".format(model_train_rmse))
    print(" - MAE: {:.4f}".format(model_train_mae))
    print(" - MSE: {:.4f}".format(model_train_mse))
    print(" - R2 Score: {:.4f}".format(model_train_r2))

    print("----------------------------------")

    print('Model performance for test set')

    print(" - RMSE: {:.4f}".format(model_test_rmse))
    print(" - MAE: {:.4f}".format(model_test_mae))
    print(" - MSE: {:.4f}".format(model_test_mse))
    print(" - R2 Score: {:.4f}".format(model_test_r2))

    r2_list.append(model_test_r2)

    print('=' * 35)
    print('\n')

LinearRegression
Model performance for training set
 - RMSE: 5.3231
 - MAE: 4.2667
 - MSE: 28.3349
 - R2 Score: 0.8743
----------------------------------
Model performance for test set
 - RMSE: 5.3940
 - MAE: 4.2148
 - MSE: 29.0952
 - R2 Score: 0.8804


Lasso
Model performance for training set
 - RMSE: 6.5938
 - MAE: 5.2063
 - MSE: 43.4784
 - R2 Score: 0.8071
----------------------------------
Model performance for test set
 - RMSE: 6.5197
 - MAE: 5.1579
 - MSE: 42.5064
 - R2 Score: 0.8253


Ridge
Model performance for training set
 - RMSE: 5.3233
 - MAE: 4.2650
 - MSE: 28.3378
 - R2 Score: 0.8743
----------------------------------
Model performance for test set
 - RMSE: 5.3904
 - MAE: 4.2111
 - MSE: 29.0563
 - R2 Score: 0.8806


K-Neighbor Regressor
Model performance for training set
 - RMSE: 5.7077
 - MAE: 4.5167
 - MSE: 32.5776
 - R2 Score: 0.8555
----------------------------------
Model performance for test set
 - RMSE: 7.2530
 - MAE: 5.6210
 - MSE: 52.6066
 - R2 Score: 0.7838


De

In [21]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
2,Ridge,0.880593
0,LinearRegression,0.880433
7,CatBoost Regressor,0.851632
5,Random Forest,0.850929
8,AdaBoost Regressor,0.845917
6,XGBoost Regressor,0.827797
1,Lasso,0.825320
3,K-Neighbor Regressor,0.783813
4,Decision Tree,0.721252


In [22]:
lin_model = LinearRegression(fit_intercept=True)
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
score = r2_score(y_test, y_pred)*100
print(" Accuracy of the model is %.2f" %score)


 Accuracy of the model is 88.04


In [23]:

pred_df=pd.DataFrame({'Actual Value':y_test,'Predicted Value':y_pred,'Difference':y_test-y_pred})
pred_df

,Actual Value,Predicted Value,Difference
521,91,76.387970,14.612030
737,53,58.885970,-5.885970
740,80,76.990265,3.009735
660,74,76.851804,-2.851804
411,84,87.627378,-3.627378
...,...,...,...
408,52,43.409149,8.590851
332,62,62.152214,-0.152214
208,74,67.888395,6.111605
613,65,67.022287,-2.022287
